In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torchattacks
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, roc_curve
from scipy.spatial.distance import mahalanobis
from scipy.linalg import inv
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
BATCH_SIZE = 64
EPOCHS_AT = 5  
LR = 0.001
EPSILONS = [0.0001, 0.0002, 0.0003, 0.0004, 0.001, 0.002, 0.005, 0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4]
BIM_STEPS = 10
print(f"🖥️ Device: {device}")

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
trainset = torchvision.datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
testset  = torchvision.datasets.FashionMNIST('./data', train=False, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
testloader  = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

class Model_B(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        with torch.no_grad():
            self.feat_size = self.features(torch.zeros(1, 1, 28, 28)).numel()
        self.fc1 = nn.Linear(self.feat_size, 512)
        self.fc2 = nn.Linear(512, 10)
        self.relu = nn.ReLU()
        self.feature_maps = {}

    def forward(self, x):
        x = self.features(x)
        self.feature_maps['layer2'] = x
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

def train_adversarial(model, loader, epochs, lr, eps_at, alpha_at, steps_at):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    attack = torchattacks.PGD(model, eps=eps_at, alpha=alpha_at, steps=steps_at, random_start=True)
    
    for epoch in range(epochs):
        loss_c, loss_a = 0.0, 0.0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            
            loss_clean = criterion(model(X), y)
            
            X_adv = attack(X, y)
            loss_adv = criterion(model(X_adv), y)
            
            loss = 0.5 * loss_clean + 0.5 * loss_adv
            loss.backward()
            optimizer.step()
            
            loss_c += loss_clean.item()
            loss_a += loss_adv.item()
        print(f"AT Epoch {epoch+1}/{epochs} | L_clean: {loss_c/len(loader):.4f} | L_adv: {loss_a/len(loader):.4f}")
    return model

def extract_features_and_labels(model, loader):
    model.eval()
    feats, labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            _ = model(X)
            feats.append(model.feature_maps['layer2'].cpu().numpy().reshape(X.size(0), -1))
            labels.append(y.numpy().astype(int))
    return np.vstack(feats), np.concatenate(labels)

class MahalanobisDetector:
    def __init__(self, reg_cov=1e-4):
        self.class_means = {}
        self.inv_covs = {}
        self.reg = reg_cov
    def fit(self, X, y, num_classes=10):
        for c in range(num_classes):
            mask = y == c
            feats = X[mask]
            if len(feats) == 0: continue
            self.class_means[c] = feats.mean(axis=0)
            cov = np.cov(feats, rowvar=False)
            cov += self.reg * np.eye(cov.shape[0])
            self.inv_covs[c] = inv(cov)
        return self
    def score(self, X, y):
        scores = []
        for i in range(len(y)):
            c = int(y[i])
            if c not in self.class_means:
                scores.append(1e6); continue
            diff = X[i] - self.class_means[c]
            d = mahalanobis(diff, np.zeros_like(diff), self.inv_covs[c])
            scores.append(d**2 if np.isfinite(d) else 1e6)
        return np.array(scores)

class PCADetector:
    def __init__(self, n_components=0.95):
        self.pca = PCA(n_components=n_components, svd_solver='full')
    def fit(self, X):
        self.pca.fit(X)
        return self
    def score(self, X):
        X_red = self.pca.transform(X)
        X_rec = self.pca.inverse_transform(X_red)
        return np.sum((X - X_rec) ** 2, axis=1)

class OCSVMDetector:
    def __init__(self, nu=0.01):
        self.clf = OneClassSVM(nu=nu, gamma='scale')
    def fit(self, X):
        self.clf.fit(X)
        return self
    def score(self, X):
        return -self.clf.decision_function(X)

class IFDetector:
    def __init__(self, n_estimators=100):
        self.clf = IsolationForest(n_estimators=n_estimators, contamination='auto', random_state=42, n_jobs=-1)
    def fit(self, X):
        self.clf.fit(X)
        return self
    def score(self, X):
        return -self.clf.decision_function(X)

def tpr_at_fpr(y_true, y_scores, fpr_target):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    return np.interp(fpr_target, fpr, tpr)

if __name__ == "__main__":
    print(f"\n🚀 Обучение Model_B с адверсариальным обучением (PGD-AT, eps=0.2)...")
    model = Model_B().to(device)
    
    train_adversarial(model, trainloader, EPOCHS_AT, LR, eps_at=0.2, alpha_at=0.02, steps_at=7)

    print("\n🔍 Извлечение чистых активаций и обучение детекторов...")
    X_clean, y_clean = extract_features_and_labels(model, testloader)
    
    detectors = {
        "Mahalanobis": MahalanobisDetector(reg_cov=1e-4),
        "PCA": PCADetector(n_components=0.95),
        "OCSVM": OCSVMDetector(nu=0.01),
        "IF": IFDetector(n_estimators=100)
    }
    
    scores_clean = {}
    for name, det in detectors.items():
        if name == "Mahalanobis":
            det.fit(X_clean, y_clean)
            scores_clean[name] = det.score(X_clean, y_clean)
        else:
            det.fit(X_clean)
            scores_clean[name] = det.score(X_clean)

    print(f"\n📈 Результаты (Model_B_AT, атака: BIM, шаги={BIM_STEPS}, alpha=eps/4):")
    header = f"{'ε':<8} | {'Детектор':<15} | {'AUC':<7} | {'TPR@1%':<7} | {'TPR@5%':<7} | {'TPR@10%':<7}"
    print(header)
    print("-" * len(header))

    for eps in EPSILONS:
        model.eval()
        X_adv_list, y_adv_list = [], []
        atk = torchattacks.BIM(model, eps=eps, steps=BIM_STEPS, alpha=eps/4)
        
        for X, y in testloader:
            X, y = X.to(device), y.to(device)
            X_adv = atk(X, y)  # Генерация атаки (требует градиенты)
            with torch.no_grad():
                _ = model(X_adv)
                f = model.feature_maps['layer2'].cpu().numpy().reshape(X_adv.size(0), -1)
            X_adv_list.append(f)
            y_adv_list.append(y.cpu().numpy().astype(int))
            
        X_adv = np.vstack(X_adv_list)
        y_adv = np.concatenate(y_adv_list)

        for det_name, det in detectors.items():
   
            if det_name == "Mahalanobis":
                scores_adv = det.score(X_adv, y_adv)
            else:
                scores_adv = det.score(X_adv)
            
            sc_c = scores_clean[det_name].copy()
            sc_a = scores_adv.copy()
            if np.mean(sc_c) > np.mean(sc_a):
                sc_c = -sc_c
                sc_a = -sc_a
            
            y_true = np.concatenate([np.zeros(len(sc_c)), np.ones(len(sc_a))])
            y_scores = np.concatenate([sc_c, sc_a])
            
            auc = roc_auc_score(y_true, y_scores)
            tpr1 = tpr_at_fpr(y_true, y_scores, 0.01)
            tpr5 = tpr_at_fpr(y_true, y_scores, 0.05)
            tpr10 = tpr_at_fpr(y_true, y_scores, 0.10)
            
            print(f"{eps:<8.4f} | {det_name:<15} | {auc:<7.4f} | {tpr1:<7.3f} | {tpr5:<7.3f} | {tpr10:<7.3f}")

🖥️ Device: cpu

🚀 Обучение Model_B с адверсариальным обучением (PGD-AT, eps=0.2)...
AT Epoch 1/5 | L_clean: 0.4350 | L_adv: 0.9797
AT Epoch 2/5 | L_clean: 0.2893 | L_adv: 0.7889
AT Epoch 3/5 | L_clean: 0.2491 | L_adv: 0.7383
AT Epoch 4/5 | L_clean: 0.2180 | L_adv: 0.7046
AT Epoch 5/5 | L_clean: 0.1955 | L_adv: 0.6801

🔍 Извлечение чистых активаций и обучение детекторов...

📈 Результаты (Model_B_AT, атака: BIM, шаги=10, alpha=eps/4):
ε        | Детектор        | AUC     | TPR@1%  | TPR@5%  | TPR@10%
------------------------------------------------------------------
0.0001   | Mahalanobis     | 0.5029  | 0.010   | 0.050   | 0.100  
0.0001   | PCA             | 0.5001  | 0.010   | 0.050   | 0.100  
0.0001   | OCSVM           | 0.5003  | 0.010   | 0.050   | 0.100  
0.0001   | IF              | 0.5001  | 0.010   | 0.050   | 0.100  
0.0002   | Mahalanobis     | 0.5048  | 0.010   | 0.050   | 0.100  
0.0002   | PCA             | 0.5002  | 0.010   | 0.050   | 0.100  
0.0002   | OCSVM           